# Notebook 03 — Detección de Regímenes de Mercado (HMM)

**Objetivo:** Identificar estados de mercado latentes usando un Modelo Oculto de Markov (HMM) Gaussiano sobre retornos y volatilidades realizadas. Cada estado recibe una etiqueta semántica (Crisis, Alcista, Lateral, etc.) y se calcula la probabilidad de transición entre estados.

**Metodología:**
- Se construye una matriz de características: retorno diario, volatilidad realizada 21d y 63d, VIX
- Se selecciona el número óptimo de estados (2-6) por BIC
- Cada estado se etiqueta semánticamente según sus medias en espacio estandarizado
- Se generan probabilidades forward a 22 días usando la potencia de la matriz de transición

**Outputs:**
- `hmm_model.pkl` — modelo HMM + scaler + etiquetas
- `hmm_regime_features.parquet` — régimen por día + probabilidades forward
- `hmm_state_stats.parquet` — estadísticas por estado
- `hmm_transition_matrix.parquet` — matriz de transición P[i→j]

In [1]:
%matplotlib inline
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
np.random.seed(42)

def _resolve_root():
    cwd = Path.cwd().resolve()
    for c in [cwd, cwd.parent, cwd.parent.parent]:
        if (c / 'data').exists() and (c / 'src').exists(): return c
    return cwd

ROOT          = _resolve_root()
sys.path.insert(0, str(ROOT))
PROCESSED_DIR = ROOT / 'data' / 'processed'
FIG_DIR       = ROOT / 'data' / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)
print(f'Project root: {ROOT}')

Project root: C:\Users\Usuario\Documents\TFG\tfg_xiker_code


## 1. Carga de datos y proxy de mercado

In [2]:
from src.nb03_hmm import build_market_proxy, build_hmm_features

returns = pd.read_parquet(PROCESSED_DIR / 'returns_daily.parquet')
macro   = pd.read_parquet(PROCESSED_DIR / 'macro_data.parquet')
print(f'Retornos: {returns.shape}  Macro: {macro.shape}')

mkt_ret = build_market_proxy(returns, macro)

Retornos: (4793, 33)  Macro: (4861, 3)
Market proxy: S&P 500 (4860 days)


## 2. Construcción de la matriz de características HMM

Características: retorno diario del mercado, varianza realizada a 21 y 63 días, y VIX (si disponible). Todo se estandariza con StandardScaler:

In [3]:
features, X_scaled, scaler = build_hmm_features(mkt_ret, macro, returns.index)
feat_idx = {c: i for i, c in enumerate(features.columns)}
print(f'\nMatrix de características: {features.shape}')
features.describe().round(4)

HMM feature matrix: (4731, 4)  cols=['market_return', 'realized_vol_21d', 'realized_vol_63d', 'vix']

Matrix de características: (4731, 4)


,market_return,realized_vol_21d,realized_vol_63d,vix
count,4731.0000,4731.0000,4731.0000,4731.0000
mean,0.0004,0.0002,0.0002,20.0006
std,0.0125,0.0003,0.0003,8.7967
min,-0.1159,0.0000,0.0000,9.1400
25%,-0.0041,0.0000,0.0000,14.2150
50%,0.0007,0.0001,0.0001,17.6800
75%,0.0060,0.0001,0.0002,22.9850
max,0.1356,0.0033,0.0022,82.6900


## 3. Selección del número óptimo de estados por BIC

Se entrenan HMMs con 2 a 6 estados (10 inicializaciones aleatorias por cada n). Se selecciona el modelo con menor BIC (Criterio de Información Bayesiano):

In [4]:
from src.nb03_hmm import fit_hmm_bic

model, results_df, optimal_n = fit_hmm_bic(X_scaled)
print(f'\nEstados seleccionados: {optimal_n}')
results_df[['n_states','bic','aic','log_likelihood']].round(1)


Selecting optimal number of HMM states...
  n=2: BIC=67502247.9  AIC=67501963.6


Model is not converging.  Current: -2586.7719500477824 is not greater than -2586.7719304838815. Delta is -1.9563900877983542e-05
Model is not converging.  Current: -2585.0600892042353 is not greater than -2585.0600728653358. Delta is -1.633889951335732e-05


  n=3: BIC=24460422.9  AIC=24459977.0


Model is not converging.  Current: -710.2454129844805 is not greater than -710.245292549579. Delta is -0.00012043490153246239
Model is not converging.  Current: -710.2456530134959 is not greater than -710.2455791238071. Delta is -7.388968879240565e-05


  n=4: BIC=6721154.7  AIC=6720534.4
  n=5: BIC=-5690785.5  AIC=-5691593.2
  n=6: BIC=-17674003.0  AIC=-17675011.1
Optimal states (BIC): 6

Estados seleccionados: 6


,n_states,bic,aic,log_likelihood
0,2,67502247.9,67501963.6,-7134.0
1,3,24460422.9,24459977.0,-2585.1
2,4,6721154.7,6720534.4,-710.2
3,5,-5690785.5,-5691593.2,601.5
4,6,-17674003.0,-17675011.1,1868.0


**Interpretación:** El BIC penaliza la complejidad del modelo, favoreciendo el equilibrio entre ajuste y parsimonia. Con 6 estados se obtiene el BIC mínimo, lo que indica que el mercado presenta 6 regímenes distinguibles con sus propias características de retorno y volatilidad.

## 4. Etiquetado semántico de estados

Cada estado recibe una etiqueta según sus medias en el espacio estandarizado: retorno alto/bajo, volatilidad alta/baja, VIX alto/bajo:

In [5]:
from src.nb03_hmm import compute_state_labels

state_labels = compute_state_labels(model, feat_idx)
print('Etiquetas semánticas:')
for s, lbl in sorted(state_labels.items()):
    print(f'  Estado {s}: {lbl}')

Etiquetas semánticas:
  Estado 0: Sideways Low-Vol
  Estado 1: Transition
  Estado 2: Transition
  Estado 3: Sideways Low-Vol
  Estado 4: Transition
  Estado 5: High Uncertainty


## 5. Estadísticas por estado de mercado

In [6]:
from src.nb03_hmm import compute_state_stats, compute_regime_features

regime_features, states, probs = compute_regime_features(model, X_scaled, features, state_labels)
state_stats = compute_state_stats(model, states, features, state_labels)
state_stats


Regime features: (4731, 17)  cols=['current_regime', 'current_regime_id', 'prob_transition', 'predicted_prob_bull_22d', 'predicted_prob_bear_22d', 'predicted_prob_transition_22d', 'predicted_prob_bull_21d', 'predicted_prob_bear_21d', 'predicted_prob_transition_21d', 'regime_stability', 'regime_duration', 'prob_state_0', 'prob_state_1', 'prob_state_2', 'prob_state_3', 'prob_state_4', 'prob_state_5']

State statistics:
                  Label  Count Pct_Days  AnnReturn%  AnnVol%  Sharpe  Persistence  Min_Return  Max_Return
State                                                                                                    
0      Sideways Low-Vol    729    15.4%       21.53     7.52  0.1804       0.9773    -0.01790     0.01693
1            Transition    876    18.5%        7.43    17.97  0.0261       0.9816    -0.03502     0.03531
2            Transition    953    20.1%       10.89    15.66  0.0438       0.9677    -0.04272     0.02865
3      Sideways Low-Vol   1381    29.2%       10

,Label,Count,Pct_Days,AnnReturn%,AnnVol%,Sharpe,Persistence,Min_Return,Max_Return
State,,,,,,,,,
0,Sideways Low-Vol,729,15.4%,21.53,7.52,0.1804,0.9773,-0.01790,0.01693
1,Transition,876,18.5%,7.43,17.97,0.0261,0.9816,-0.03502,0.03531
2,Transition,953,20.1%,10.89,15.66,0.0438,0.9677,-0.04272,0.02865
3,Sideways Low-Vol,1381,29.2%,10.87,11.69,0.0586,0.9729,-0.03218,0.02531
4,Transition,536,11.3%,11.94,26.03,0.0289,0.9717,-0.05054,0.05350
5,High Uncertainty,256,5.4%,-24.13,54.47,-0.0279,0.9844,-0.11589,0.13558


**Interpretación:** El estado *Lateral Moderado* (S3) es el más frecuente (29,2% del tiempo) y representa mercados sin tendencia clara y volatilidad contenida. El estado *Crisis / Alta Tensión* (S5) tiene rentabilidad negativa y alta volatilidad (5,4% del período), capturando episodios como 2008, 2011, COVID-2020. El estado *Alcista Estable* (S0, 15,4%) agrupa los períodos de mercado alcista sostenido con baja volatilidad. La persistencia alta (diagonal de la matriz de transición >0,95) indica que los regímenes son estables: una vez que el mercado entra en un estado, tiende a permanecer en él varios días.

## 6. Visualización de regímenes temporales

In [7]:
from src.nb03_hmm import plot_regimes

plot_regimes(features, states, state_labels, probs, FIG_DIR)
plt.show()

Saved: C:\Users\Usuario\Documents\TFG\tfg_xiker_code\data\results\figures\03_hmm_regimes_v5.png


**Interpretación:** El panel superior muestra el S&P 500 con el fondo coloreado según el régimen vigente cada día. El panel inferior muestra las probabilidades suavizadas (soft assignments) de cada estado apiladas, que permiten capturar la incertidumbre en los períodos de transición entre regímenes. Los episodios de crisis históricos (2008, COVID-2020) son capturados por el estado S5 (Crisis / Alta Tensión), validando la capacidad discriminativa del HMM.

## 7. Matriz de transición entre estados

In [8]:
from src.nb03_hmm import plot_transition_heatmap
import pandas as pd

trans_df = pd.DataFrame(model.transmat_,
    index=[f'{s}:{state_labels[s][:10]}' for s in range(model.n_components)],
    columns=[f'{s}:{state_labels[s][:10]}' for s in range(model.n_components)])
print('Matriz de transición P[i→j]:')
display(trans_df.round(3))

plot_transition_heatmap(model, state_labels, FIG_DIR)
plt.show()

Matriz de transición P[i→j]:


,0:Sideways L,1:Transition,2:Transition,3:Sideways L,4:Transition,5:High Uncer
0:Sideways L,0.977,0.000,0.000,0.023,0.000,0.000
1:Transition,0.000,0.982,0.013,0.000,0.006,0.000
2:Transition,0.000,0.005,0.968,0.021,0.006,0.000
3:Sideways L,0.012,0.000,0.015,0.973,0.000,0.000
4:Transition,0.000,0.021,0.000,0.000,0.972,0.007
5:High Uncer,0.000,0.000,0.000,0.000,0.016,0.984


Saved: C:\Users\Usuario\Documents\TFG\tfg_xiker_code\data\results\figures\03_hmm_transition_v5.png


**Interpretación:** La diagonal alta (>0.95) confirma que los regímenes son persistentes: una vez el mercado entra en un estado, tiende a permanecer en él varios días. Las transiciones fuera de la diagonal representan cambios de régimen. La probabilidad de pasar de Lateral Moderado (S3) directamente a Crisis / Alta Tensión (S5) es baja, lo que valida el carácter progresivo de los movimientos de mercado.

## 8. Características de régimen para downstream

Se generan probabilidades forward a 22 días multiplicando las probabilidades actuales por la potencia 22 de la matriz de transición: P(régimen en 22d) = p_actual × T²²

In [9]:
print(f'Características de régimen: {regime_features.shape}')
print(f'Columnas: {list(regime_features.columns)}')
regime_features.tail(5)

Características de régimen: (4731, 17)
Columnas: ['current_regime', 'current_regime_id', 'prob_transition', 'predicted_prob_bull_22d', 'predicted_prob_bear_22d', 'predicted_prob_transition_22d', 'predicted_prob_bull_21d', 'predicted_prob_bear_21d', 'predicted_prob_transition_21d', 'regime_stability', 'regime_duration', 'prob_state_0', 'prob_state_1', 'prob_state_2', 'prob_state_3', 'prob_state_4', 'prob_state_5']


,current_regime,current_regime_id,prob_transition,predicted_prob_bull_22d,predicted_prob_bear_22d,predicted_prob_transition_22d,predicted_prob_bull_21d,predicted_prob_bear_21d,predicted_prob_transition_21d,regime_stability,regime_duration,prob_state_0,prob_state_1,prob_state_2,prob_state_3,prob_state_4,prob_state_5
Date,,,,,,,,,,,,,,,,,
2026-04-23,Transition,2,0.03234,0.0,0.0,1.0,0.0,0.0,1.0,0.96766,34,2.146761e-42,0.000002,0.999998,3.563286e-09,2.236929e-10,1.339487e-21
2026-04-24,Transition,2,0.03234,0.0,0.0,1.0,0.0,0.0,1.0,0.96766,35,8.932185e-43,0.000002,0.999998,2.804556e-09,1.378328e-10,2.254103e-21
2026-04-27,Transition,2,0.03234,0.0,0.0,1.0,0.0,0.0,1.0,0.96766,36,1.169342e-34,0.000001,0.999999,3.589116e-08,2.044102e-10,1.251730e-19
2026-04-28,Transition,2,0.03234,0.0,0.0,1.0,0.0,0.0,1.0,0.96766,37,7.389775e-28,0.000003,0.999997,1.121994e-07,2.246360e-08,3.745485e-17
2026-04-29,Transition,2,0.03234,0.0,0.0,1.0,0.0,0.0,1.0,0.96766,38,1.567391e-21,0.000107,0.999880,3.546674e-06,9.484213e-06,8.944640e-13


## 9. Guardado de outputs

In [10]:
from src.nb03_hmm import learn_dag, save_all_outputs

learn_dag(features, states, state_labels, PROCESSED_DIR)
save_all_outputs(model, scaler, state_labels, features, X_scaled,
                 regime_features, state_stats, PROCESSED_DIR)

print(f'\n✓ NB03 COMPLETO — {model.n_components} estados, {len(features)} días (2010-2024)')
for s, lbl in sorted(state_labels.items()):
    print(f'  Estado {s}: {lbl}')

pgmpy not available — DAG learning skipped

Saved: hmm_model.pkl | hmm_regime_features.parquet | hmm_state_stats.parquet
       hmm_transition_matrix.parquet | hmm_state_probabilities.parquet
       hmm_features_raw.parquet | hmm_features_scaled.parquet | hmm_state_labels.json

✓ NB03 COMPLETO — 6 estados, 4731 días (2010-2024)
  Estado 0: Sideways Low-Vol
  Estado 1: Transition
  Estado 2: Transition
  Estado 3: Sideways Low-Vol
  Estado 4: Transition
  Estado 5: High Uncertainty
